In [1]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests
%pip install polars

  Using cached openmeteo_requests-1.7.5-py3-none-any.whl (7.1 kB)
  Using cached niquests-3.18.2-py3-none-any.whl (207 kB)
  Using cached openmeteo_sdk-1.25.0-py3-none-any.whl (19 kB)
     -------------------------------------- 153.9/153.9 kB 3.1 MB/s eta 0:00:00
  Using cached urllib3_future-2.17.903-py3-none-any.whl (710 kB)
  Using cached wassima-2.0.5-py3-none-any.whl (138 kB)
  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl (30 kB)
  Using cached h11-0.16.0-py3-none-any.whl (37 kB)
  Using cached jh2-5.0.10-cp37-abi3-win_amd64.whl (246 kB)
  Using cached qh3-1.6.0-cp37-abi3-win_amd64.whl (2.0 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached requests_cache-1.3.1-py3-none-any.whl (70 kB)
  Using cached retry_requests-2.0.0-py3-none-any.whl (15 kB)
  Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
  Using cached cattrs-26.1.0-py3-none-any.whl (73 kB)
     ---------------------------------------- 64.7/64.7 kB 1.8 MB/s eta 0:00:00
  Using cached url_normalize-2.2.1-py3-none-any.whl (14 kB)
     -------------------------------------- 131.6/131.6 kB 1.9 MB/s eta 0:00:00
     ---------------------------------------- 71.0/71.0 kB ? eta 0:00:00
     -------------------------------------- 153.7/153.7 kB 9.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     -------------------------------------- 824.0/824.0 kB 8.7 MB/s eta 0:00:00
     --------------------------------------- 47.0/47.0 MB 29.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import openmeteo_requests
import polars as pl
import requests_cache
from retry_requests import retry
from datetime import datetime, timedelta, timezone

In [5]:
# api config
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

url = "https://api.open-meteo.com/v1/forecast"

Functionise the below to make easier

In [6]:
# Call config
# Oxford
_latitude = 52.52
_longitude = 13.41

# The order of variables in hourly or daily is important to assign them correctly below
# Would be better to define a dictionary of the variables then pass in the names/order to use in later arguments. Then there is no risk of assigning values of one and mislabelling them.
var_list = [
    "temperature_2m", 
    "relative_humidity_2m", 
    "apparent_temperature", 
    "precipitation_probability", 
    "precipitation", "rain", 
    "showers", "snowfall", 
    "snow_depth", 
    "weather_code", 
    "cloud_cover", 
    "wind_speed_10m"
]

params = {
	"latitude": _latitude,
	"longitude": _longitude,
	"hourly": var_list,
}



In [ ]:
# api call
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
hourly = response.Hourly()

# Setting up the datetime length to create the dataframe with
start = datetime.fromtimestamp(hourly.Time(), timezone.utc)
end = datetime.fromtimestamp(hourly.TimeEnd(), timezone.utc)
freq = timedelta(seconds = hourly.Interval())

# Dataframe base
# get the columns before populating with metrics
df_base = pl.select(
    lat = response.Latitude(),
    long = response.Longitude(),
    extract_date = datetime.fromtimestamp(hourly.Time(), timezone.utc).date(),
    extract_time = datetime.now().time(),
    fc_datetime = pl.datetime_range(start, end, freq, closed="left")
).with_columns(
    fc_date = pl.col('fc_datetime').dt.date(),
    fc_hour = pl.col('fc_datetime').dt.strftime("%H")
)

col_exprs = {var: hourly.Variables(var_list.index(var)).ValuesAsNumpy() for var in var_list}

df_metrics = pl.DataFrame(
    col_exprs
)

df_raw = pl.concat([df_base, df_metrics], how='horizontal')

To do:
- Motherduck staging table
- Motherduck main table - recency rank
- Motherduck light table for latest values only (where recency rank = 1)